# Primera LGD con historia separada

**Disponibilidad:** estas puertas pertenecen al candidato de desarrollo y todavía no están publicadas en PyPI 2.16.0. Este cuaderno se ejecutó con el [wheel candidato verificado](https://github.com/nexolabs-gh/bayesrisk/actions/runs/39523257538), disponible en el artefacto `candidate-distributions-with-evidence`. Instalar la versión publicada en PyPI no incorpora estas funciones.

Recorrido experimental; requiere `bayesrisk[scoring,excel]`. Datos inventados incluidos en el paquete: 400 operaciones, 300 train, 80 evaluation y 20 abiertas excluidas. Dos operaciones por deudor dentro de una muestra; ningún deudor cruza las muestras elegibles. Objetivo en [0,1] con extremos 0 y 1 y dos covariables numéricas crudas. Las abiertas conservan objetivo vacío.

La institución debe declarar población, disponibilidad y muestras antes del ajuste. El motor no verifica suficiencia semántica ni temporalidad. Las métricas y la media de referencia son descriptivas; no validan calidad ni transportabilidad.

## Ajustar y evaluar

Los seis roles no se infieren. El ID de operación se declara por separado. Sólo train ajusta el modelo; evaluation no cambia sus coeficientes.

In [1]:
from pathlib import Path

import pandas as pd

from bayesrisk import Lgd, Provision
from bayesrisk.ui.datasets import materialize

historia = materialize("lgd_historia_sintetica", workdir=Path("datos"))
lgd = Lgd(historia, target="lgd_observada", features=["deuda_ingreso", "antiguedad_meses"],
          entity="deudor", sample="muestra", default="incumplio", resolved="resuelto",
          id="loan_id", purpose="Demostrar el flujo con datos inventados",
          assumptions=["Historia sintética con muestras y resultado declarados"],
          limitations=["No valida calidad ni transportabilidad"], formats=["csv", "xlsx"])
lgd.run(raise_on_error=True)
lgd.summary("lgd_training")

── Historia de operaciones ──
Archivo: datos/datasets/lgd_historia_sintetica.parquet (copia en bayesrisk-runs/lgd/input/data-<hash>.parquet)
400 filas · 7 columnas
Identificador: loan_id (índice del archivo)
Se conserva toda la historia; las exclusiones se cuentan antes del ajuste
Evidencia de la corrida: bayesrisk-runs/lgd/run
── Entrenamiento LGD ──
300 operaciones de entrenamiento · 80 de evaluación
20 operaciones excluidas, conservadas en la historia
2 covariables crudas; respuesta fraccional con intercepto
Sólo entrenamiento ajusta el predictor y la media de referencia
Las métricas describen la muestra; no certifican calidad ni transportabilidad
⚠ Declaración aportada; suficiencia y temporalidad no verificadas
⚠ Propósito: Demostrar el flujo con datos inventados
Supuestos: Historia sintética con muestras y resultado declarados
Limitaciones: No valida calidad ni transportabilidad
⚠ Procesos abiertos excluidos; representatividad y ausencia de sesgo de selección no demostradas
── Inf

StageSummary(stage='lgd_training', label='Entrenamiento LGD', lines=('300 operaciones de entrenamiento · 80 de evaluación', '20 operaciones excluidas, conservadas en la historia', '2 covariables crudas; respuesta fraccional con intercepto', 'Sólo entrenamiento ajusta el predictor y la media de referencia', 'Las métricas describen la muestra; no certifican calidad ni transportabilidad'), alerts=('Declaración aportada; suficiencia y temporalidad no verificadas', 'Propósito: Demostrar el flujo con datos inventados/nSupuestos: Historia sintética con muestras y resultado declarados/nLimitaciones: No valida calidad ni transportabilidad', 'Procesos abiertos excluidos; representatividad y ausencia de sesgo de selección no demostradas'), table=         Muestra  ... Raíz del error cuadrático medio
0  Entrenamiento  ...                        0.171575
1  Entrenamiento  ...                        0.196807
2     Evaluación  ...                        0.156369
3     Evaluación  ...                        0.176759
[4 rows x 8 columns], formats={})

## Aplicar a otra cartera sin resultado observado

La cartera sintética tiene 6.000 operaciones. La PD constante es ilustrativa. Se elimina la columna LGD: el predictor usa sólo covariables crudas y no vuelve a ajustar. La provisión conserva la fórmula grupal, con detalle prorrateado. Separar muestras declaradas no certifica independencia del pipeline completo.

El CSV y el libro modelo_lgd exportan una tabla global reutilizable, sin llave de alineación; los textos codificados se conservan. No se cargan Study/joblib por HTTP. Los ZIP conservan historia y aplicación por separado. Las huellas y el ID de ajuste que aparecen abajo identifican esta ejecución; cambian con sus entradas o una corrida nueva.

In [2]:
cartera = pd.read_parquet(materialize("provision_interna_generica", workdir=Path("datos")))
cartera = cartera.drop(columns="lgd")
cartera["pd_declarada"] = 0.02  # Sólo ilustración; no es una PD estimada.
provision = Provision(cartera, as_of="as_of_date", portfolio="portfolio",
                      exposure="exposure_amount", pd="pd_declarada", lgd=lgd.model_,
                      id="loan_id", formats=["csv"])
provision.run(raise_on_error=True)
lgd.export_excel()
lgd.export("historia-lgd.zip")
provision.export("aplicacion-lgd.zip")
provision.summary()

── Cartera ──
Archivo: DataFrame en memoria, guardado como bayesrisk-runs/provision/input/data-<hash>.parquet
6.000 filas · 12 columnas
Esquema inferido: 12 columnas
Identificador: loan_id (índice del archivo)
PD externa declarada; esta corrida no ajusta ni calibra la PD
Evidencia de la corrida: bayesrisk-runs/provision/run
── Provisión interna ──
6.000 operaciones · 3 grupos
Exposición: 8.129.332.047 · Provisión: 58.845.153
Agrupación: banda de score
PD externa declarada: no se ajustó ni calibró en esta corrida
El detalle prorratea el total grupal por exposición; no suma productos individuales
La severidad se predice con un modelo de historia separada, sin reajustarlo con esta cartera.
⚠ Declaración aportada; suficiencia y temporalidad no verificadas
⚠ Propósito: Demostrar el flujo con datos inventados
Supuestos: Historia sintética con muestras y resultado declarados
Limitaciones: No valida calidad ni transportabilidad
⚠ La aplicación no acredita independencia ni transportabilidad; no

Provisión interna total,58.845.153
Exposición total,8.129.332.047
Operaciones,6.000
Grupos,3
Cobertura,"0,72 %"
